GSE146409: Liver Tumor Microenvironment

source: https://ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE146409


In [10]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings


In [3]:
!curl -sL -o GSE146409_counts.csv.gz "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE146nnn/GSE146409/suppl/GSE146409_UMI_counts_of_filtered_cells.csv.gz" && gunzip -k GSE146409_counts.csv.gz && curl -sL -o GSE146409_metadata.csv.gz "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE146nnn/GSE146409/suppl/GSE146409_metadata_of_filtered_cells.csv.gz" && gunzip -k GSE146409_metadata.csv.gz && echo "Done: $(wc -l < GSE146409_counts.csv) lines counts, $(wc -l < GSE146409_metadata.csv) lines metadata"


Done: 15163 lines counts, 7948 lines metadata


In [4]:
counts = pd.read_csv('GSE146409_counts.csv', index_col=0)
print(f'Raw counts: {counts.shape[0]} genes × {counts.shape[1]} cells')

meta = pd.read_csv('GSE146409_metadata.csv', index_col='cell')
print(f'Metadata: {meta.shape[0]} cells × {meta.shape[1]} columns')
assert list(meta.index) == list(counts.columns), 'Mismatch!'
adata = sc.AnnData(X=counts.T.values.astype(np.float32),
                   obs=meta,
                   var=pd.DataFrame(index=counts.index))
adata.var_names_make_unique()
print(f'AnnData: {adata.shape}')
print(f'\nGround truth distribution:')
print(adata.obs['mergeCarcinoma'].value_counts())


Raw counts: 15162 genes × 7947 cells
Metadata: 7947 cells × 12 columns
AnnData: (7947, 15162)
Ground truth distribution:
mergeCarcinoma
Carcinoma         1639
SAMs              1092
Kupffer cells      694
LVEC               673
CAFs               591
cDC2               565
LSEC               398
Hepatocytes        391
TM1                363
T cells            301
LVECt              282
Stellate cells     233
vSMC               208
B cells            187
Pericytes          152
Proliferation      117
cDC1                61
Name: count, dtype: int64


In [5]:
adata.obs['ground_truth_malignant'] = (adata.obs['mergeCarcinoma'] == 'Carcinoma').astype(int)
print(f'Malignant: {adata.obs["ground_truth_malignant"].sum()} / {len(adata)} ({adata.obs["ground_truth_malignant"].mean()*100:.1f}%)')


Malignant: 1639 / 7947 (20.6%)


In [7]:
from aeacus import Profiler

aeacus_profiler = Profiler(
    test_input=adata.copy(),
    norm_type='cpm_log1p'
)
aeacus_profiler.load()
aeacus_adata = aeacus_profiler.profile()

print('aeacus results:')
print(aeacus_adata.obs[['malignancy_call', 'malignancy_score']].head(10))

malignant_count = (aeacus_adata.obs["malignancy_call"] == "Malignant").sum()
print(f'\nMalignant calls: {malignant_count} / {len(aeacus_adata)}')


Model features: 3778
Missing features: 80 (2.12%)
aeacus results:
               malignancy_call  malignancy_score
cell
h2_AAACACACGTG          Normal          0.033532
h2_AAACTGCACAA          Normal          0.015242
h2_AAACCGTAACT          Normal          0.077908
h2_AAACAGCTCAA          Normal          0.019991
h2_AAACTGTCACG          Normal          0.018632
h2_AAACCCTAAGG          Normal          0.016237
h2_AAACGGCGATT          Normal          0.047967
h2_AAACACTGCAG       Malignant          0.814125
h2_AAACCGCATAT          Normal          0.013473
h2_AAACGCCTACA          Normal          0.011371
Malignant calls: 807 / 7947
